In [6]:
import os
import sys
import numpy as np
import moeabench as mb

sys.path.append(os.path.abspath("."))
from src.meamt_dinamico import MEAMT_DIN
from src.mochila import MochilaMultiobjetivo

MAX_FES = 300000
NOBJ = 5
MIN_TABLES_MEAMT = 30

pop_meamt = MIN_TABLES_MEAMT * (2 ** NOBJ)
gen_meamt = MAX_FES // pop_meamt

prob = mb.mops.DTLZ3(M=NOBJ, N=NOBJ + 10 - 1)

exp1 = mb.experiment()
exp1.moea = mb.moeas.NSGA3(population=300, generations=1000)
exp1.mop = prob

exp2 = mb.experiment()
exp2.moea = MEAMT_DIN(population=pop_meamt, generations= gen_meamt)
exp2.mop = prob


exp1.run(repeat=1) 
exp2.run(repeat=1) 


### Running **exp1**

  Run 1/1:   0%|          | 0/1000 [00:00<?, ?gen/s]

### Running **exp2**

  Run 1/1:   0%|          | 0/312 [00:00<?, ?gen/s]

In [7]:
Z_ref = prob.pf(1500)
nadir = np.max(Z_ref, axis=0)
limite = nadir * 1.5

F_atual1 = np.array(exp1[0].pop().objs)
mascara1 = np.all(F_atual1 <= limite, axis=1)
F_final1 = F_atual1[mascara1]

F_atual2 = np.array(exp2[0].pop().objs)
mascara2 = np.all(F_atual2 <= limite, axis=1)
F_final2 = F_atual2[mascara2]


mb.view.topology(F_final1, F_final2, show_gt=True, gt=exp1.optimal_front())
hv1 = mb.metrics.hypervolume(F_final1, ref=Z_ref)
hv2 = mb.metrics.hypervolume(F_final2, ref=Z_ref)
hv1.report()
hv2.report()


Computing Hypervolume (Array):   0%

Computing Hypervolume (Array):   0%

================================================
### Metric Report: Hypervolume (Raw) (Array)
> **Physical Objective Space**: How much objective space has been physically conquered within the global search boundaries established in this session?

#### Final Performance (Last Gen)
- **Mean**: 0.4003
- **StdDev**: 0.0000
- **Best**: 0.4003

#### Search Dynamics
- **Runs**: 1
- **Generations**: 1
- **Stability**: High (CV=0.0000 < 0.05)


================================================
### Metric Report: Hypervolume (Raw) (Array)
> **Physical Objective Space**: How much objective space has been physically conquered within the global search boundaries established in this session?

#### Final Performance (Last Gen)
- **Mean**: 1.2975
- **StdDev**: 0.0000
- **Best**: 1.2975

#### Search Dynamics
- **Runs**: 1
- **Generations**: 1
- **Stability**: High (CV=0.0000 < 0.05)


In [8]:
prob2 = MochilaMultiobjetivo(n_obj=NOBJ, n_itens=100)
exp3 = mb.experiment()
exp3.moea = mb.moeas.NSGA3(population=300, generations=1000)
exp3.mop = prob2

exp4 = mb.experiment()
exp4.moea = MEAMT_DIN(population=pop_meamt, generations= gen_meamt)
exp4.mop = prob2

exp3.run(repeat=1)
exp4.run(repeat=1)

### Running **exp3**

  Run 1/1:   0%|          | 0/1000 [00:00<?, ?gen/s]

### Running **exp4**

  Run 1/1:   0%|          | 0/312 [00:00<?, ?gen/s]

In [9]:
import gurobipy as gp
from gurobipy import GRB

modelo = gp.Model("MochilaMultiobjetivo")
modelo.Params.LogToConsole = 0  # Desativa a saída de log do Gurobi
grid_steps = 40
x = modelo.addVars(prob2.N, vtype=GRB.BINARY, name="x")

for j in range(prob2.M):
    peso_total_j = gp.quicksum(prob2.weights[i, j] * x[i] for i in range(prob2.N))
    modelo.addConstr(peso_total_j <= prob2.capacities[j], name=f"capacidade_{j}")
    
lucros = []
for j in range(prob2.M):
    lucro_total_j = gp.quicksum(prob2.values[i, j] * x[i] for i in range(prob2.N))
    lucros.append(lucro_total_j)

modelo.setObjective(lucros[0], GRB.MAXIMIZE)  # Define o objetivo como maximizar o lucro do primeiro objetivo

restricao_eps2 = modelo.addConstr(lucros[1] >= 0, name="restricao_eps2")  # Adiciona a restrição para o segundo objetivo
restricao_eps3 = modelo.addConstr(lucros[2] >= 0, name="restricao_eps3")  # Adiciona a restrição para o terceiro objetivo

passo_eps2 = prob2.max_profits[1] / grid_steps
passo_eps3 = prob2.max_profits[2] / grid_steps

fronteira_pareto = []
solucoes_vistas = set()

for step2 in range(grid_steps + 1):
    eps2 = step2 * passo_eps2
    restricao_eps2.rhs = eps2
    for step3 in range(grid_steps + 1):
        eps3 = step3 * passo_eps3
        
        restricao_eps3.rhs = eps3
        
        modelo.optimize()
        
        if modelo.status == GRB.OPTIMAL:
            x_bin = np.array([x[i].X for i in range(prob2.N)])
            assinatura = tuple(np.round(x_bin).astype(int))  # Arredonda para evitar problemas de precisão
            if assinatura not in solucoes_vistas:
                solucoes_vistas.add(assinatura)
                fit = prob2.evaluation(x_bin)['F'][0]
                fronteira_pareto.append(fit)
                
mb.view.topology(exp3, exp4, show_gt=True, gt=fronteira_pareto)

Set parameter LogToConsole to value 0


In [ ]:
fronteira_pareto = np.array(fronteira_pareto)
hv3 = mb.metrics.hypervolume(exp3, ref=[exp3, exp4])
hv4 = mb.metrics.hypervolume(exp4, ref=[exp3, exp4])

hv3.report()
hv4.report()

Computing Hypervolume (exp3):   0%

Computing Hypervolume (exp4):   0%

===============================================
### Metric Report: Hypervolume (Raw) (exp3)
> **Physical Objective Space**: How much objective space has been physically conquered within the global search boundaries established in this session?

#### Final Performance (Last Gen)
- **Mean**: 1.1489
- **StdDev**: 0.0000
- **Best**: 1.1489

#### Search Dynamics
- **Runs**: 1
- **Generations**: 1000
- **Stability**: High (CV=0.0000 < 0.05)


===============================================
### Metric Report: Hypervolume (Raw) (exp4)
> **Physical Objective Space**: How much objective space has been physically conquered within the global search boundaries established in this session?

#### Final Performance (Last Gen)
- **Mean**: 0.8379
- **StdDev**: 0.0000
- **Best**: 0.8379

#### Search Dynamics
- **Runs**: 1
- **Generations**: 313
- **Stability**: High (CV=0.0000 < 0.05)
